In [8]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd
import rioxarray as rioxr
from shapely import box

import pystac_client
import planetary_computer

from IPython.display import Image


sys.path.append("../utils")

pd.set_option("display.max_columns", None)


In [9]:
in19 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2019.geojson"
)
in20 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2020.geojson"
)
in21 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2021.geojson"
)
in22 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2022.geojson"
)
in23 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2023.geojson"
)


In [10]:
# structuret

colnames_23 = {
    "system_created_at": "system_cre",
    "address_suite": "address_su",
    "address_thoroughfare": "address_th",
    "address_locality": "address_lo",
    "address_sub_admin_area": "address__2",
    "address_postal_code": "address_po",
    "address_full": "address_fu",
    "inspectiondate_calculate": "inspection",
    "structuretype": "structuret",
    "address_admin_area": "address_ad",
    "address_country": "address_co",
}

in23 = in23.rename(columns=colnames_23)


In [11]:
inspections = pd.concat([in19, in20, in21, in22, in23], axis=0, ignore_index=True)


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [16]:
inspections["inspection_id"] = np.arange(1, len(inspections) + 1)

/Users/ryangreen/.conda/envs/prg/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


In [17]:
inspections = inspections[["inspection_id", "structuret"]]


In [7]:
structuretype_codes = (
    inspections[["structuret"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

structuretype_codes["structure_code"] = range(100, 100 + len(structuretype_codes))

structuretype_codes = structuretype_codes.rename(columns = {'structuret': 'structure_type'})

structuretype_codes


NameError: name 'inspections' is not defined

In [19]:
structuretype_codes.to_csv(
    "/capstone/wildfire_prep/data/metadata/structuretype_codes.csv",
    index=False,
)


In [27]:
inspections

,inspection_id,structuret
0,1,Utility or Miscellaneous Structure > 120 sqft
1,2,Single Family Residence Multi Story
2,3,Single Family Residence Multi Story
3,4,Utility or Miscellaneous Structure > 120 sqft
4,5,Single Family Residence Multi Story
...,...,...
67575,67576,Single Family Residence Multi Story
67576,67577,None
67577,67578,Mobile Home Double Wide
67578,67579,Mobile Home Double Wide


In [37]:
inspections = inspections.merge(
    structuretype_codes[["structuret", "structure_code"]],
    on="structuret",
    how="left",
).drop(columns=["structuret"])
inspections

,inspection_id,structure_code
0,1,100
1,2,101
2,3,101
3,4,100
4,5,101
...,...,...
67575,67576,101
67576,67577,116
67577,67578,106
67578,67579,106


In [38]:
inspections.to_csv(
    "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspections_id_structure_type.csv",
    index=False,
)
